In [1]:
import pandas as pd
import numpy as np
import os
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# -----------------------------
# Load data
# -----------------------------
def load_data():

    print("Loading data...")

    train = pd.read_csv("../data/train.csv", encoding="cp932")
    test = pd.read_csv("../data/test.csv", encoding="cp932")

    print("Train shape:", train.shape)
    print("Test shape:", test.shape)

    return train, test


# -----------------------------
# Prepare features
# -----------------------------
def prepare_features(train, test):

    spectral_cols = [
        c for c in train.columns
        if c not in ["sample number", "species number", "樹種", "含水率"]
    ]

    X = train[spectral_cols].copy()
    X_test = test[spectral_cols].copy()

    # Add species number as feature
    X["species"] = train["species number"]
    X_test["species"] = test["species number"]

    y = train["含水率"].values

    print("Number of features:", X.shape[1])

    return X.values, y, X_test.values


# -----------------------------
# Scale features
# -----------------------------
def scale_features(X, X_test):

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)
    X_test_scaled = scaler.transform(X_test)

    print("Feature scaling complete")

    return X_scaled, X_test_scaled


# -----------------------------
# Hyperparameter search
# -----------------------------
def hyperparameter_search(X, y):

    print("\nStarting ElasticNet hyperparameter search...\n")

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    # MUCH wider search
    alphas = np.logspace(-4, 1, 20)
    l1_ratios = np.linspace(0.05, 0.95, 10)

    best_score = np.inf
    best_params = None

    for alpha in alphas:
        for l1 in l1_ratios:

            rmse_scores = []

            for train_idx, val_idx in kf.split(X):

                X_train, X_val = X[train_idx], X[val_idx]
                y_train, y_val = y[train_idx], y[val_idx]

                model = ElasticNet(
                    alpha=alpha,
                    l1_ratio=l1,
                    max_iter=100000,
                    tol=1e-3,
                    random_state=42
                )

                model.fit(X_train, y_train)

                preds = model.predict(X_val)

                rmse = np.sqrt(mean_squared_error(y_val, preds))
                rmse_scores.append(rmse)

            mean_rmse = np.mean(rmse_scores)

            print(f"alpha={alpha:.5f}  l1_ratio={l1:.2f}  RMSE={mean_rmse:.4f}")

            if mean_rmse < best_score:
                best_score = mean_rmse
                best_params = (alpha, l1)

    print("\nBest parameters found:")
    print("alpha =", best_params[0])
    print("l1_ratio =", best_params[1])
    print("Best CV RMSE =", best_score)

    return best_params


# -----------------------------
# Train final model
# -----------------------------
def train_final_model(X, y, X_test, alpha, l1_ratio):

    print("\nTraining final model...")

    model = ElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=100000,
        tol=1e-3,
        random_state=42
    )

    model.fit(X, y)

    preds = model.predict(X_test)

    print("Sample predictions:", preds[:10])

    return preds


# -----------------------------
# Save submission
# -----------------------------
def save_submission(test, preds):

    os.makedirs("../submissions", exist_ok=True)

    submission = pd.DataFrame({
        "sample number": test["sample number"],
        "含水率": preds
    })

    experiment_name = "exp15_elasticnet_species_wide_search_20260324"

    output_path = f"../submissions/{experiment_name}.csv"

    submission.to_csv(output_path, index=False, header=False)

    print("\nSubmission saved to:", output_path)

    check = pd.read_csv(output_path, header=None)
    print(check.head())


# -----------------------------
# Main pipeline
# -----------------------------
def main():

    train, test = load_data()

    X, y, X_test = prepare_features(train, test)

    X, X_test = scale_features(X, X_test)

    best_alpha, best_l1 = hyperparameter_search(X, y)

    preds = train_final_model(X, y, X_test, best_alpha, best_l1)

    save_submission(test, preds)


main()

Loading data...
Train shape: (1322, 1559)
Test shape: (550, 1558)
Number of features: 1556
Feature scaling complete

Starting ElasticNet hyperparameter search...

alpha=0.00010  l1_ratio=0.05  RMSE=12.7094
alpha=0.00010  l1_ratio=0.15  RMSE=12.8466
alpha=0.00010  l1_ratio=0.25  RMSE=13.0194
alpha=0.00010  l1_ratio=0.35  RMSE=13.2386
alpha=0.00010  l1_ratio=0.45  RMSE=13.5212
alpha=0.00010  l1_ratio=0.55  RMSE=13.8912
alpha=0.00010  l1_ratio=0.65  RMSE=14.3840
alpha=0.00010  l1_ratio=0.75  RMSE=15.0102
alpha=0.00010  l1_ratio=0.85  RMSE=15.8152
alpha=0.00010  l1_ratio=0.95  RMSE=16.9493
alpha=0.00018  l1_ratio=0.05  RMSE=12.3805
alpha=0.00018  l1_ratio=0.15  RMSE=12.3752
alpha=0.00018  l1_ratio=0.25  RMSE=12.4002
alpha=0.00018  l1_ratio=0.35  RMSE=12.4700
alpha=0.00018  l1_ratio=0.45  RMSE=12.6049
alpha=0.00018  l1_ratio=0.55  RMSE=12.8281
alpha=0.00018  l1_ratio=0.65  RMSE=13.1744
alpha=0.00018  l1_ratio=0.75  RMSE=13.7267
alpha=0.00018  l1_ratio=0.85  RMSE=14.6600
alpha=0.00018  l1_ra